# Check the recorded step counts of the February re-run objects

The objects shared on November 6, 2026 (`2026_11_06_qc_regen_nick_andy_coords/`) come with `step_counts.tsv`: one row
per sample with the reads, cells or nuclei at every step. This notebook recounts every number from its primary source,
not from the script that wrote the table, and compares the two.

| column | recounted from |
|---|---|
| final nuclei, positioned (Andy), positioned (Nick) | the shared objects (`qc_passed_{mouse,human}.h5ad`) |
| pass QC thresholds, doublets removed | each sample's QC log (`2026_11_06_qc_regen/logs/01_qc_<sample>.log`) |
| CellBender nuclei | the barcodes in each CellBender output (`xmens_inputs_regen/<sample>/`) |
| Cell Ranger cells, reads | each lane's Cell Ranger output (`velocity/<lane>/cellranger/outs/`) |

Everything is read-only. Run it with the rsc_uv environment's Python; the checks take a few seconds.

In [1]:
from pathlib import Path
import re

import h5py
import pandas as pd
from anndata.io import read_elem

NH = Path("/n/lab_storage/eddy_lab/users/nhilgert")
NS = Path("/n/netscratch/eddy_lab/Everyone/nhilgert")

SHARED = NS / "2026_11_06_qc_regen_nick_andy_coords"  # the shared objects and their step_counts.tsv
QC_LOGS = NS / "2026_11_06_qc_regen" / "logs"  # logs of the QC run that made the objects
CELLBENDER = NH / "decidualization" / "xmens_inputs_regen"  # July 2026 CellBender run
CELLRANGER = NH / "decidualization" / "revisions" / "velocity"  # February 2026 Cell Ranger run

COUNT_COLUMNS = [
    "reads",
    "Cell Ranger cells",
    "CellBender nuclei",
    "pass QC thresholds",
    "doublets removed",
    "final nuclei",
    "positioned (Andy)",
    "positioned (Nick)",
]

## The recorded table

In [2]:
recorded = pd.read_csv(SHARED / "step_counts.tsv", sep="\t")
recorded_samples = recorded[recorded["sample"] != "total"].set_index("sample")
recorded_totals = recorded[recorded["sample"] == "total"].set_index("species")
recorded

,species,sample,lane,reads,Cell Ranger cells,CellBender nuclei,pass QC thresholds,doublets removed,final nuclei,positioned (Andy),positioned (Nick)
0,mouse,1_dpi_early_mck1,mck_1,191981288,3765,4106,3567,70,3497,1853,1787
1,mouse,1_dpi_late_mck12a,mck_12a,586181794,10513,9884,8893,1125,7768,4478,4478
2,mouse,1_dpi_late_mck12b,mck_12b,364217847,8941,8770,7852,815,7037,3129,3129
3,mouse,2_dpi_mck15,mck_15,230403475,6405,7372,5513,360,5153,2978,2978
4,mouse,3_dpi_mck10a,mck_10a,384897848,14509,12807,9877,1470,8407,4121,4121
5,mouse,3_dpi_mck10b,mck_10b,790967709,14445,10949,9902,1656,8246,4625,4625
6,mouse,4_dpi_bleeding_mck11a,mck_11a,1039108587,13901,11849,10610,1864,8746,6145,6143
7,mouse,uninduced_mck8a,mck_8a,837511274,14040,15733,13882,1855,12027,4104,4108
8,mouse,uninduced_mck8b,mck_8b,827006620,11770,11507,10227,1801,8426,3569,3570
9,human,menstrual_mck5a,mck_5a,763434770,17638,17553,15300,2558,12742,7591,7594


## Final and positioned nuclei, from the shared objects

A nucleus counts as positioned when both of its coordinates are present. Only `obs` is read, so the count matrices
stay on disk.

In [3]:
def count_nuclei(path):
    """Nuclei per sample in one shared object, and how many each coordinate set places."""
    with h5py.File(path, "r") as f:
        obs = read_elem(f["obs"])

    placed_andy = obs["x_um_dbscan_andy"].notna() & obs["y_um_dbscan_andy"].notna()
    placed_nick = obs["x_um_dbscan_nick"].notna() & obs["y_um_dbscan_nick"].notna()
    return pd.DataFrame({
        "final nuclei": obs.groupby("sample", observed=True).size(),
        "positioned (Andy)": placed_andy.groupby(obs["sample"], observed=True).sum(),
        "positioned (Nick)": placed_nick.groupby(obs["sample"], observed=True).sum(),
    })


from_objects = pd.concat(
    [count_nuclei(SHARED / f"qc_passed_{species}.h5ad") for species in ("mouse", "human")]
)
from_objects

,final nuclei,positioned (Andy),positioned (Nick)
sample,,,
1_dpi_early_mck1,3497,1853,1787
1_dpi_late_mck12a,7768,4478,4478
1_dpi_late_mck12b,7037,3129,3129
2_dpi_mck15,5153,2978,2978
3_dpi_mck10a,8407,4121,4121
3_dpi_mck10b,8246,4625,4625
4_dpi_bleeding_mck11a,8746,6145,6143
uninduced_mck8a,12027,4104,4108
uninduced_mck8b,8426,3569,3570


## Threshold and doublet counts, from the QC logs

Each sample's log records the nuclei CellBender passed on, how many passed the thresholds, how many DoubletDetection
flagged, and how many remained.

In [4]:
def read_qc_log(sample):
    """The QC log's counts for one sample."""
    text = (QC_LOGS / f"01_qc_{sample}.log").read_text()
    return {
        "CellBender nuclei (log)": int(re.search(r"loaded (\d+) cells", text).group(1)),
        "pass QC thresholds": int(re.search(r"after threshold QC: (\d+)/", text).group(1)),
        "doublets removed": int(re.search(r"flagged (\d+) doublets", text).group(1)),
        "final nuclei (log)": int(re.search(r"final: (\d+) cells", text).group(1)),
    }


from_logs = pd.DataFrame({sample: read_qc_log(sample) for sample in recorded_samples.index}).T
from_logs

,CellBender nuclei (log),pass QC thresholds,doublets removed,final nuclei (log)
1_dpi_early_mck1,4106,3567,70,3497
1_dpi_late_mck12a,9884,8893,1125,7768
1_dpi_late_mck12b,8770,7852,815,7037
2_dpi_mck15,7372,5513,360,5153
3_dpi_mck10a,12807,9877,1470,8407
3_dpi_mck10b,10949,9902,1656,8246
4_dpi_bleeding_mck11a,11849,10610,1864,8746
uninduced_mck8a,15733,13882,1855,12027
uninduced_mck8b,11507,10227,1801,8426
menstrual_mck5a,17553,15300,2558,12742


## CellBender nuclei, from the CellBender outputs

In [5]:
def count_cellbender_nuclei(sample):
    """Barcodes CellBender called as cells: the columns of its filtered matrix."""
    with h5py.File(CELLBENDER / sample / f"{sample}_cellbender_filtered.h5", "r") as f:
        return len(f["matrix/barcodes"])


from_cellbender = pd.Series(
    {sample: count_cellbender_nuclei(sample) for sample in recorded_samples.index},
    name="CellBender nuclei",
)
from_cellbender

1_dpi_early_mck1          4106
1_dpi_late_mck12a         9884
1_dpi_late_mck12b         8770
2_dpi_mck15               7372
3_dpi_mck10a             12807
3_dpi_mck10b             10949
4_dpi_bleeding_mck11a    11849
uninduced_mck8a          15733
uninduced_mck8b          11507
menstrual_mck5a          17553
menstrual_mck5b          18270
menstrual_mck5c          16059
secretory_mck6a          17930
secretory_mck6b          17500
secretory_mck6c           9933
Name: CellBender nuclei, dtype: int64

## Cell Ranger cells and reads, from the Cell Ranger outputs

Cell Ranger cells are the barcodes in each lane's filtered matrix. `metrics_summary.csv` reports the read count, and
the cell count again as Estimated Number of Cells.

In [6]:
def read_cellranger(lane):
    """Reads and called cells for one lane."""
    outs = CELLRANGER / lane / "cellranger" / "outs"
    metrics = pd.read_csv(outs / "metrics_summary.csv", thousands=",")
    with h5py.File(outs / "filtered_feature_bc_matrix.h5", "r") as f:
        n_barcodes = len(f["matrix/barcodes"])
    return {
        "reads": int(metrics["Number of Reads"][0]),
        "Cell Ranger cells": n_barcodes,
        "estimated cells (metrics)": int(metrics["Estimated Number of Cells"][0]),
    }


from_cellranger = pd.DataFrame(
    {sample: read_cellranger(lane) for sample, lane in recorded_samples["lane"].items()}
).T
from_cellranger

,reads,Cell Ranger cells,estimated cells (metrics)
1_dpi_early_mck1,191981288,3765,3765
1_dpi_late_mck12a,586181794,10513,10513
1_dpi_late_mck12b,364217847,8941,8941
2_dpi_mck15,230403475,6405,6405
3_dpi_mck10a,384897848,14509,14509
3_dpi_mck10b,790967709,14445,14445
4_dpi_bleeding_mck11a,1039108587,13901,13901
uninduced_mck8a,837511274,14040,14040
uninduced_mck8b,827006620,11770,11770
menstrual_mck5a,763434770,17638,17638


## Compare

Where two sources report the same number, they must agree first. Then every recounted number is compared with the
recorded table, sample by sample and for the species totals.

In [7]:
recounted = pd.concat(
    [from_cellranger, from_cellbender, from_logs, from_objects], axis=1
).loc[recorded_samples.index]

consistency = {
    "CellBender output = QC log 'loaded'": (
        recounted["CellBender nuclei"] == recounted["CellBender nuclei (log)"]
    ),
    "object nuclei = QC log 'final'": recounted["final nuclei"] == recounted["final nuclei (log)"],
    "thresholds - doublets = final": (
        recounted["pass QC thresholds"] - recounted["doublets removed"] == recounted["final nuclei"]
    ),
    "filtered barcodes = estimated cells": (
        recounted["Cell Ranger cells"] == recounted["estimated cells (metrics)"]
    ),
}
for check, agrees in consistency.items():
    print(f"{check}: {int(agrees.sum())} of {len(agrees)} samples")

CellBender output = QC log 'loaded': 15 of 15 samples
object nuclei = QC log 'final': 15 of 15 samples
thresholds - doublets = final: 15 of 15 samples
filtered barcodes = estimated cells: 15 of 15 samples


In [8]:
per_sample = recounted[COUNT_COLUMNS].astype(int)
differs = per_sample != recorded_samples[COUNT_COLUMNS]
print(f"per sample: {int((~differs).values.sum())} of {differs.size} numbers match the recorded table")

totals = per_sample.groupby(recorded_samples["species"]).sum()
totals_differ = totals != recorded_totals.loc[totals.index, COUNT_COLUMNS]
print(f"species totals: {int((~totals_differ).values.sum())} of {totals_differ.size} numbers match")

mismatched = differs.any(axis=1)
if mismatched.any():
    side_by_side = pd.concat(
        {"recorded": recorded_samples[COUNT_COLUMNS], "recounted": per_sample}, axis=1
    )
    print(side_by_side[mismatched])
else:
    print("no differences")

per sample: 120 of 120 numbers match the recorded table
species totals: 16 of 16 numbers match
no differences
